# ДЗ-16 (часть 1): LoRA fine-tuning чат-ассистента (Qwen2.5-1.5B)

Дообучаем **Qwen2.5-1.5B-Instruct** под качественные диалоги методом **QLoRA**
(LoRA поверх 4-битной модели) на сабсете датасета **lmsys/lmsys-chat-1m**.

## Зачем LoRA / PEFT
Полный fine-tuning LLM меняет все веса (миллиарды параметров) — это дорого по памяти и времени.
**PEFT** (Parameter-Efficient Fine-Tuning) обучает лишь малую добавку. **LoRA** вставляет в слои
внимания обучаемые низкоранговые матрицы `A·B` (ранг `r`), а исходные веса замораживает —
обучается ~0.1–1% параметров. **QLoRA** дополнительно квантует базовую модель в 4 бита, чтобы
влезть в один бесплатный GPU (Colab T4, 16 ГБ) или в скромную локальную видеокарту.

> ⚠️ **Нужен NVIDIA GPU.** `bitsandbytes` (4-бит) работает только на CUDA. Если локально CUDA нет
> (`torch.cuda.is_available()` вернёт `False`) — запускай в **Google Colab**
> (Runtime → Change runtime type → T4 GPU).

## Почему ядро падало и что изменено

Ноутбук подстраивается под объём VRAM автоматически (флаг `LOW_VRAM`). На картах <6 ГБ
(проверено на Quadro P2000, 4 ГБ) исходная конфигурация приводила к
`The Kernel crashed while executing code` и к очень медленным ячейкам. Причины и исправления:

1. **Тихий сброс в системную RAM — главная причина и крашей, и тормозов.**
   На Windows драйвер NVIDIA при нехватке VRAM не выдаёт ошибку, а незаметно вытесняет
   тензоры в обычную память через PCIe. Замер: `batch=2, seq_len=1024` "успешно" запрашивал
   **10.9 ГБ на карте с 4 ГБ**, и один шаг занимал **82 секунды**; затем ядро умирало.
   Исправление — `torch.cuda.set_per_process_memory_fraction(0.92)`: теперь вместо свопа
   мы получаем честный `CUDA out of memory` в traceback, который не убивает ядро.

2. **fp32-эмбеддинги съедали 890 МБ.** `prepare_model_for_kbit_training` апкастит
   неквантованные веса в fp32, а у Qwen2.5 матрица `embed_tokens` (151936 × 1536, привязана
   к `lm_head`) в fp32 весит 890 МБ — 22% всей карты. Мы её не обучаем, поэтому возвращаем
   в fp16. Без этого даже `batch=1, seq_len=256` не помещался в 4 ГБ.

3. **Узкое место — логиты, а не веса модели.** `transformers` в функции лосса делает
   `logits.float()`, храня fp16-логиты + fp32-копию + градиент: при `batch·seq = 2048`
   и словаре 151936 это ~2.9 ГБ **только на вычисление лосса**. Поэтому на низкой VRAM
   уменьшено `batch·seq`, а эффективный размер батча добирается накоплением градиента.

4. **Повторный запуск ячейки загружал вторую копию модели** поверх первой (+1.1 ГБ) —
   теперь ячейка загрузки идемпотентна и сначала освобождает VRAM.

**Результат замера на P2000:** `1.75 с/шаг` при пике `~3.2 ГБ` вместо `82 с/шаг` и краша —
примерно **47× быстрее**. Датасет дополнительно кэшируется на диск, что экономит ~80 секунд
при каждом перезапуске ядра.

> ℹ️ `packing=True` (склейка коротких диалогов, убирает padding) мог бы дать ещё прирост,
> но без Flash Attention диалоги внутри блока начинают «видеть» друг друга. Flash Attention
> требует Ampere+, поэтому на Pascal packing намеренно выключен.

## Шаг 1. Установка (в Colab)
Раскомментируй и выполни в Colab. Локально с CUDA ставь из `requirements.txt`.

In [6]:
# !pip install -q -U torch transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), (
    "CUDA-GPU не найден. Запусти в Colab с T4 (Runtime -> Change runtime type -> GPU)."
)

# ВАЖНО (Windows + NVIDIA): при нехватке VRAM драйвер по умолчанию НЕ падает с ошибкой,
# а незаметно вытесняет тензоры в системную RAM через PCIe ("system memory fallback").
# Обучение при этом не крашится сразу, но шаг замедляется в десятки раз (замерено: 82 с/шаг
# вместо 1.75 с/шаг), а затем ядро Jupyter умирает без внятного сообщения —
# это и есть "The Kernel crashed while executing code".
# Ставим жёсткий потолок: теперь вместо тихого свопа мы получим честный CUDA OOM,
# который видно в traceback и который не убивает ядро.
torch.cuda.set_per_process_memory_fraction(0.92)

props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1024**3
IS_PASCAL = props.major < 7          # P2000 = CC 6.1: нет bf16 и int8 tensor cores
LOW_VRAM = VRAM_GB < 6

print(f"GPU: {props.name} | {VRAM_GB:.1f} ГБ | CC {props.major}.{props.minor}")
print(f"Режим: {'low-VRAM (<6 ГБ)' if LOW_VRAM else 'обычный'}, "
      f"{'Pascal — только fp16' if IS_PASCAL else 'поддерживается bf16'}")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"


def free_vram(*objs):
    """Освободить VRAM: удалить объекты, собрать мусор, вернуть кэш аллокатора драйверу."""
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def vram_report(tag=""):
    used = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"[VRAM]{' ' + tag if tag else ''} занято {used:.2f} ГБ, пик {peak:.2f} ГБ / {VRAM_GB:.1f} ГБ")

## Шаг 2. Загрузка модели в 4-битном виде (QLoRA)

In [ ]:
# =====================================================================
# Загрузка модели в 4 бита (QLoRA). Ячейка идемпотентна:
# повторный запуск не создаёт вторую копию модели в VRAM.
# =====================================================================

# bitsandbytes 8-bit (LLM.int8()) требует int8 tensor cores (CC >= 7.5, Turing+).
# На Pascal (P2000, CC 6.1) быстрого пути нет, а llm_int8_enable_fp32_cpu_offload
# гоняет данные между GPU и CPU — шаг обучения превращается в многоминутное ожидание.
# 4-битный NF4 не требует int8-ядер и работает на Pascal нормально.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16, не bf16 — Pascal не поддерживает bf16
    bnb_4bit_use_double_quant=True,
)

# Если модель уже в памяти (повторный прогон ячейки) — сначала освобождаем VRAM.
# Без этого вторая загрузка добавляет ~1.1 ГБ поверх первой и почти гарантирует OOM на 4 ГБ.
for _name in ("trainer", "model"):
    if _name in globals():
        print(f"Освобождаю предыдущий объект '{_name}' перед перезагрузкой...")
        globals().pop(_name)
free_vram()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Загрузка токенизатора и модели {MODEL_NAME}...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    dtype=torch.float16,
    # Явное device_map={"": 0} вместо "auto": при нехватке места "auto" молча раскидывает
    # часть слоёв на CPU/диск, и тогда каждый шаг тащит веса по PCIe (очень медленно).
    # Лучше честный OOM, чем незаметный оффлоад.
    device_map={"": 0},
)
model.config.use_cache = False   # обязательно при обучении с gradient checkpointing

vram_report("после загрузки модели")
print("✅ Модель загружена в 4-битном режиме (NF4).")

## Шаг 3. Baseline ДО обучения
Сохраним ответ исходной модели на тестовый вопрос, чтобы потом сравнить с дообученной.

In [ ]:
from contextlib import nullcontext


def chat(model, question: str, max_new_tokens: int = 200) -> str:
    """Генерация ответа по chat-шаблону Qwen, стабильная на CUDA и с квантованием."""
    messages = [{"role": "user", "content": question}]

    # Формируем правильный промпт со всеми разметками
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Токенизируем и явно отправляем на то же устройство, где находится первый слой модели
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # use_cache=True только для генерации (ускоряет вывод в разы).
    # Во время обучения он выключен — несовместим с gradient checkpointing.
    was_cache = model.config.use_cache
    model.config.use_cache = True
    model.eval()

    # После обучения LoRA-адаптеры остаются в fp32, а база — в fp16, и обычный forward
    # падает с "expected mat1 and mat2 to have the same dtype".
    # autocast приводит типы к общему знаменателю прямо во время матричных операций.
    autocast = torch.autocast("cuda", dtype=torch.float16) if model.device.type == "cuda" \
        else nullcontext()
    try:
        with torch.inference_mode(), autocast:
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        prompt_length = inputs.input_ids.shape[1]
        answer = tokenizer.decode(out[0][prompt_length:], skip_special_tokens=True).strip()
    finally:
        # Возвращаем режим и освобождаем KV-кэш: иначе он остаётся в VRAM и
        # "съедает" память, которая нужна следующему шагу обучения.
        model.config.use_cache = was_cache
        model.train()
        free_vram()
    return answer


# Тестируем базовую модель
TEST_Q = "Объясни простыми словами, чем отличается обучение с учителем от обучения без учителя."
baseline_answer = chat(model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)
vram_report("после baseline-генерации")

## Шаг 4. Датасет: lmsys-chat-1m

`lmsys/lmsys-chat-1m` — **gated**: зайди на
[страницу датасета](https://huggingface.co/datasets/lmsys/lmsys-chat-1m), прими условия и
залогинься токеном (`HF_TOKEN`). Берём небольшой сабсет через streaming (не качаем весь 1M).

Если доступа к lmsys нет — функция автоматически переключится на открытый
`HuggingFaceH4/ultrachat_200k`.

In [ ]:
import os
import json
from pathlib import Path
from itertools import islice
from datasets import load_dataset, Dataset
from huggingface_hub import login

if os.getenv("HF_TOKEN"):
    login(os.environ["HF_TOKEN"])

N_SAMPLES = 2000        # сабсет для демонстрации; увеличь для лучшего качества
MAX_TURNS = 6           # ограничим длину диалогов

# Стриминг 2000 диалогов с HF занимает ~80 секунд и повторяется при КАЖДОМ перезапуске ядра.
# Кэшируем результат на диск — повторный прогон ячейки становится мгновенным.
CACHE_FILE = Path(f"dialogues_cache_{N_SAMPLES}_{MAX_TURNS}.json")


def to_messages_lmsys(row):
    # в lmsys поле 'conversation' = [{'role': 'user'/'assistant', 'content': ...}, ...]
    return [{"role": m["role"], "content": m["content"]} for m in row["conversation"][:MAX_TURNS]]


def to_messages_ultrachat(row):
    return [{"role": m["role"], "content": m["content"]} for m in row["messages"][:MAX_TURNS]]


def download_chat_subset():
    try:
        ds = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True)
        rows = [to_messages_lmsys(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из lmsys-chat-1m")
    except Exception as e:
        print(f"lmsys недоступен ({e}). Переключаюсь на ultrachat_200k.")
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)
        rows = [to_messages_ultrachat(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из ultrachat_200k")
    return rows


def load_chat_subset():
    if CACHE_FILE.exists():
        rows = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
        print(f"Взято из локального кэша {CACHE_FILE} — {len(rows)} диалогов (сеть не нужна)")
    else:
        rows = download_chat_subset()
        CACHE_FILE.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
        print(f"Сохранено в кэш {CACHE_FILE}")
    # оставляем только корректные диалоги (начинается с user, есть ответ ассистента)
    return [r for r in rows if len(r) >= 2 and r[0]["role"] == "user"]


dialogues = load_chat_subset()
print("Диалогов после фильтрации:", len(dialogues))
print("Пример диалога:", dialogues[0][:2])

## Шаг 5. Форматирование под chat-шаблон
SFTTrainer обучается на готовом тексте. Превращаем каждый диалог в строку через
`apply_chat_template` — так модель учится в том же формате, в котором её потом спрашивают.

In [11]:
def format_example(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [{"text": format_example(d)} for d in dialogues]
train_ds = Dataset.from_list(train_texts)
print("Обучающих примеров:", len(train_ds))
print("\n--- Пример отформатированного текста ---\n", train_ds[0]["text"][:400])

Обучающих примеров: 2000

--- Пример отформатированного текста ---
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
how can identity protection services help protect me against identity theft<|im_end|>
<|im_start|>assistant
Identity protection services can help protect you against identity theft in several ways:

1. Monitoring: Many identity protection services monitor your credit reports, public r


## Шаг 6. Конфиг LoRA и обучение (SFTTrainer)

Параметры LoRA:
- `r` — ранг добавок (8/16/32): больше → выразительнее и больше обучаемых параметров;
- `lora_alpha` — масштаб добавок (обычно 2·r);
- `target_modules` — в какие слои вставлять LoRA (проекции attention + MLP);
- `lora_dropout` — регуляризация.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Перед обучением освобождаем всё, что осталось от baseline-генерации (KV-кэш и т.п.).
free_vram()

model = prepare_model_for_kbit_training(
    model,
    # use_reentrant=False — современная реализация gradient checkpointing.
    # Со старой (True) PEFT-адаптеры иногда не получают градиент и обучение молча "не идёт".
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# --- КЛЮЧЕВАЯ ОПТИМИЗАЦИЯ ПАМЯТИ -------------------------------------------------
# prepare_model_for_kbit_training апкастит НЕ-квантованные веса в fp32. У Qwen2.5
# embed_tokens имеет размер 151936 x 1536 и в fp32 занимает 890 МБ — это 22% от 4 ГБ,
# причём lm_head к нему привязан (tied weights). В fp16 те же веса весят 445 МБ.
# Мы их не обучаем (LoRA живёт в attention/MLP), поэтому fp32 здесь не нужен.
# Замер: без этого фикса даже bs=1/seq=256 не влезает в 4 ГБ.
if LOW_VRAM:
    for _n, _p in model.named_parameters():
        if _p.dtype == torch.float32 and "embed_tokens" in _n:
            _p.data = _p.data.to(torch.float16)
            _p.requires_grad_(False)
    free_vram()
    vram_report("после fp16-эмбеддингов")

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

# --- Подбор длины и батча под объём VRAM -----------------------------------------
# Главный потребитель памяти на train-шаге — НЕ веса модели, а тензор логитов:
# transformers в ForCausalLMLoss делает logits.float(), то есть держит fp16-логиты
# ПЛЮС их fp32-копию плюс градиент. Для батча B и длины L это B*L*151936*(2+4+4) байт.
# При B*L = 2048 (было: batch=2, seq=1024) это ~2.9 ГБ только на лосс — отсюда и краши.
# Держим произведение batch*seq небольшим, а эффективный батч добираем накоплением.
if LOW_VRAM:                      # 4 ГБ (P2000): замерено ~1.8 с/шаг, пик ~3.2 ГБ
    MAX_LEN, BATCH, ACCUM = 256, 1, 8
else:                             # T4 16 ГБ и подобные
    MAX_LEN, BATCH, ACCUM = 1024, 2, 4

print(f"Конфиг: max_length={MAX_LEN}, batch={BATCH}, accum={ACCUM} "
      f"(эффективный батч = {BATCH * ACCUM}, токенов на шаг = {BATCH * MAX_LEN})")

sft_config = SFTConfig(
    output_dir="qwen2.5-1.5b-lora",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    max_steps=60,
    learning_rate=2e-4,
    logging_steps=10,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",          # экономия памяти на состояниях оптимизатора

    fp16=True,                         # Pascal не поддерживает bf16 — строго fp16
    bf16=False,

    max_length=MAX_LEN,
    # ВНИМАНИЕ про packing: он склеивает несколько диалогов в один блок и убирает padding,
    # НО без Flash Attention соседние диалоги внутри блока "видят" друг друга
    # (cross-contamination) — trl прямо предупреждает об этом при запуске.
    # Flash Attention требует Ampere+ и на Pascal недоступен, поэтому packing выключен:
    # корректность важнее скорости. На современной карте (Ampere+) с установленным
    # flash-attn его можно включить — это заметно ускорит обучение.
    packing=False,
    dataset_num_proc=1,                # на Windows многопроцессность datasets часто виснет
    dataloader_num_workers=0,          # 0 — на Windows spawn-воркеры только замедляют старт

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Периодически возвращаем кэш аллокатора драйверу — снижает фрагментацию,
    # из-за которой длинный прогон мог упасть по OOM ближе к концу.
    torch_empty_cache_steps=10,

    save_strategy="no",                # 60 шагов — промежуточные чекпоинты не нужны
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

# --- ОБЯЗАТЕЛЬНО ДЛЯ PASCAL ------------------------------------------------------
# SFTTrainer в конструкторе безусловно переводит ВСЕ обучаемые веса QLoRA в bf16
# (следуя рекомендации статьи QLoRA), и отключить это нечем: autocast_adapter_dtype
# для квантованных моделей пока не поддерживается (peft#2889).
# На Pascal аппаратного bf16 нет, и обучение падает на первом же шаге:
#   RuntimeError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'
# (fp16-GradScaler не умеет работать с bf16-градиентами).
#
# Проверяем именно compute capability, а НЕ torch.cuda.is_bf16_supported(): на P2000
# последняя возвращает True, потому что учитывает медленную программную эмуляцию.
if IS_PASCAL:
    _fixed = 0
    for _p in trainer.model.parameters():
        if _p.requires_grad and _p.dtype == torch.bfloat16:
            _p.data = _p.data.to(torch.float32)
            _fixed += 1
    print(f"Pascal: {_fixed} обучаемых тензоров возвращены из bf16 в fp32")

trainer.model.print_trainable_parameters()

print("Запуск обучения...")
trainer.train()
vram_report("после обучения")

## Шаг 7. Сравнение ПОСЛЕ обучения
Тот же вопрос — но теперь отвечает модель с обученным LoRA-адаптером.

In [ ]:
ft_answer = chat(trainer.model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)
print("\n=== ПОСЛЕ fine-tuning ===\n", ft_answer)

## Шаг 8. Сохранение адаптера
Сохраняем только LoRA-адаптер (несколько МБ). В части 2 (`agent_demo.ipynb`) подгрузим его
поверх базовой модели.

In [ ]:
ADAPTER_DIR = "qwen2.5-1.5b-lora-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Адаптер сохранён в", ADAPTER_DIR)

# (опционально) слить адаптер в полную модель для удобного инференса:
# from peft import PeftModel
# merged = trainer.model.merge_and_unload()
# merged.save_pretrained("qwen2.5-1.5b-merged")

## Выводы
- **LoRA/PEFT** позволил адаптировать модель, обучив доли процента параметров — это влезло
  в бесплатный GPU благодаря **QLoRA** (4-бит).
- Сравнение «до/после» на одном вопросе демонстрирует сдвиг стиля ответов к обучающим данным.
- Для реального качества: больше шагов (`max_steps`/эпохи), больше данных, валидация и подбор `r`.
- Дальше — `agent_demo.ipynb`: подключаем адаптер и даём модели **инструменты** (часть 2).